In [ ]:
!unzip labqar_pipeline.zip -d /content/
%cd /content/labqar_pipeline

Archive:  labqar_pipeline.zip
  inflating: /content/RAG_DESIGN_JUSTIFICATION.md  
  inflating: /content/balance_dataset.py  
  inflating: /content/dataset_builder.py  
  inflating: /content/range_checker.py  
  inflating: /content/requirements.txt  
  inflating: /content/step1_build_registry.py  
  inflating: /content/step2_build_corpus.py  
  inflating: /content/step3_retriever.py  
  inflating: /content/step3b_embeddings.py  
  inflating: /content/step3c_vector_store.py  
  inflating: /content/step3d_hybrid_retriever.py  
  inflating: /content/step4_range_checker_rag.py  
  inflating: /content/step5_evaluate.py  
  inflating: /content/step5b_generate_eval_queries.py  
  inflating: /content/step5c_evaluate_rag.py  
  inflating: /content/step6_llm_explain.py  
  inflating: /content/step6b_freetext_rag_qa.py  
  inflating: /content/step7_hybrid_checker.py  
  inflating: /content/artifacts/step1_reference_registry.csv  
  inflating: /content/artifacts/step1_reference_registry.json  
  in

In [ ]:
!pip install -q -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [ ]:
!python step1_build_registry.py
!python step2_build_corpus.py
!python step3b_embeddings.py
!python step3c_vector_store.py


Registry rows built: 550 (parse failures: 0)

Set_2 label audit: 20/550 (3.6%) raw 'Answer' labels disagree with reference_range-derived ground truth.
  ID 6: value=0.61 range=[0.2,0.43] raw='Normal' corrected='High'
  ID 11: value=65.19 range=[17.4,None] raw='High' corrected='Normal'
  ID 13: value=26.1 range=[6.64,None] raw='High' corrected='Normal'
  ID 14: value=0.19 range=[0.05,None] raw='High' corrected='Normal'
  ID 42: value=135.23 range=[35.0,90.0] raw='Normal' corrected='High'
  ... full list in step1_set2_corrected_labels.json

Wrote:
  artifacts/step1_reference_registry.json/.csv  (550 rows)
  artifacts/step1_set2_corrected_labels.json     (550 rows)
Wrote 550 documents to /content/artifacts/step2_corpus.jsonl

Sample document:
{
  "doc_id": "labqar_1",
  "text": "Reference range for Acetaminophen (specimen: Serum, plasma; gender: all; age group: all): between 70.0 and 200.0 μmol/L.",
  "metadata": {
    "parameter": "Acetaminophen",
    "parameter_norm": "acetaminophen",
 

In [ ]:
!python step5_evaluate.py
!python step5b_generate_eval_queries.py
!python step5c_evaluate_rag.py


SET 1 -- RETRIEVAL EVALUATION
Total queries:         550
Exact structured hit:  550 (100.0%)
Fell back to fuzzy/other: 0

SET 2 -- CLASSIFICATION EVALUATION (deterministic range_checker)
Total queries: 550
Accuracy vs RAW  'Answer' field:       519/550 (94.4%)
Accuracy vs CORRECTED gold label:      539/550 (98.0%)
(The corrected label is the right one to trust -- see step1_build_registry.py.)
Generated 200 template-based eval queries from a sample of 100 LabQAR rows -> /content/artifacts/step5b_eval_queries.json
(These still contain the literal parameter name -- good for a sanity check, not a real test of semantic understanding. See HARD_QUERIES / step5b_hard_queries.json.)

Wrote 16 hand-written synonym/paraphrase queries (no literal parameter-name overlap) -> /content/artifacts/step5b_hard_queries.json

Sample template queries:
  [115] 'what is a normal Calcium level in serum?'  (gold: Calcium)
  [115] 'is my Calcium result within the reference range in serum?'  (gold: Calcium)
  [ 2

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = ''
!python step6_llm_explain.py
!python step6b_freetext_rag_qa.py



ALT = 52.0 U/L (all) -> High
  ALT was 52.0 U/L, above the expected reference range (reference upper limit: 40.0 U/L).

Hemoglobin = 111.0 g/L (Female) -> Low
  Hemoglobin was 111.0 g/L, below the expected reference range (reference lower limit: 120.0 g/L).

Acetaminophen = 341.62 μmol/L (all) -> High
  Acetaminophen was 341.62 μmol/L, above the expected reference range (reference upper limit: 200.0 μmol/L).
Indexed 550 documents into Chroma collection 'labqar_reference_ranges' at /content/artifacts/chroma_db
Loading weights: 100% 103/103 [00:00<00:00, 9852.97it/s]
/content/step3b_embeddings.py:70: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self._model.get_sentence_embedding_dimension()

Q: what's a healthy fasting glucose range for an adult?
A: An error occurred while generating the answer.
Retrieved:
  [0.345] Glucose
  [0.242] Glucose
  [0.154] Insulin

Q: is a TSH of 6.2 mIU/L normal for an adult?
A: An 

In [ ]:
!git config --global user.email "berraegcin@gmail.com"
!git config --global user.name "berraEgcin"

In [ ]:
!git branch

fatal: not a git repository (or any of the parent directories): .git


In [ ]:
from step7_hybrid_checker import HybridRangeChecker
from step4_range_checker_rag import RangeCheckerRAG
from step6_llm_explain import Explainer
from step6b_freetext_rag_qa import RagQA

def run_tests():
    print("Loading components...\n")

    # Init modules
    hybrid_checker = HybridRangeChecker()
    rag_checker = RangeCheckerRAG() # Sadece RAG tabanlı açıklamayı beslemek için
    explainer = Explainer() # GOOGLE_API_KEY yoksa şablon metin döner
    qa = RagQA()

    print("-" * 50)
    print("TEST 1: Structured Data Check")
    print("-" * 50)
    test_cases = [
        ("ALT", 52.0, "U/L", "Female"),
        ("Hemoglobin", 11.1, "g/dL", "Male")
    ]

    for param, val, unit, gender in test_cases:
        # Evaluate via hybrid (CSV)
        status = hybrid_checker.evaluate(param, val, gender=gender)
        print(f"Parameter: {param} | Value: {val} {unit} | Gender: {gender}")
        print(f"Decision (CSV): {status}")

        # Evaluate via LabQAR
        rag_result = rag_checker.evaluate(param, val, gender=gender)
        summary = explainer.explain(param, val, unit, rag_result)
        print(f"LLM Explanation: {summary}\n")

    print("-" * 50)
    print("TEST 2: Free-Text RAG QA")
    print("-" * 50)
    soru = "what is a normal fasting glucose range for an adult?"
    cevap = qa.answer(soru, top_k=2)
    print(f"Question: {soru}")
    print(f"Answer:\n{cevap['answer']}\n")

    print("-" * 50)
    print("TEST 3: Audit Report")
    print("-" * 50)
    # Print mismatch audit
    hybrid_checker.print_audit_report()

run_tests()

Sistem bileşenleri yükleniyor...

Indexed 550 documents into Chroma collection 'labqar_reference_ranges' at /content/artifacts/chroma_db


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/content/step3b_embeddings.py:70: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self._model.get_sentence_embedding_dimension()


--------------------------------------------------
TEST 1: Yapısal Veri Kontrolü ve Açıklama
--------------------------------------------------
Parametre: ALT | Değer: 52.0 U/L | Cinsiyet: Female
Karar (CSV'den): Normal
LLM Açıklaması: ALT was 52.0 U/L, above the expected reference range (reference upper limit: 40.0 U/L).

Parametre: Hemoglobin | Değer: 11.1 g/dL | Cinsiyet: Male
Karar (CSV'den): Low
LLM Açıklaması: Hemoglobin was 11.1 g/dL, below the expected reference range (reference lower limit: 135.0 g/L).

--------------------------------------------------
TEST 2: Serbest Metin RAG Soru-Cevap
--------------------------------------------------
Soru: what is a normal fasting glucose range for an adult?
Cevap:
An error occurred while generating the answer.

--------------------------------------------------
TEST 3: Denetim (Audit) Raporu
--------------------------------------------------
HYBRID CHECKER AUDIT REPORT (CSV=primary/authoritative vs LabQAR=secondary)
Total evaluations: 2